## Statistiques sur les indices de sécheresse
Ce fichier est basé sur l'analyse des indices de sécheresse produits en fonction du recensement agricole 2025 (dataframe "parcelles") et les régions agricoles selon le projet RISC,

### Structure générale

1. Statistiques d'indice par parcelles agricoles
2. Corrélations entre indices
3. Statistiques régionales
4. Statistiques pour exploitations RISC
5. Statistiques sur les cultures
6. Tests statistiques
    a. Statistiques descritives par groupe
    b. ANOVA et Eta-squared
    c Test de Tukey HSD post-hoc car ANOVA trop significatif pour toutes les variables

### 1. Calcul des statistiques d'indice par parcelles agricoles

In [ ]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd

def add_zonal_mean(gdf, raster_path, new_col):
    """
    Calcule la moyenne d'un raster sur chaque géométrie
    et ajoute le résultat comme nouvelle colonne.
    """
    stats = zonal_stats(
        gdf,
        raster_path,
        stats="median",
        nodata=None,
        geojson_out=False
    )

    gdf[new_col] = [s["median"] for s in stats]
    return gdf


In [ ]:
# Calcul idx_eq_gwl2 mediane
parcelles = gpd.read_file("../data/raw/RISC/2025_RISC_Recensement_agricole.shp")

parcelles = add_zonal_mean(
    parcelles,
    "../data/processed/Indices finaux/Composites/indice_eq_gwl2_norm.tif",
    "idx_eq_gwl2"
)



In [ ]:
# Calcul idx_eq_ref91 mediane
parcelles = add_zonal_mean(
    parcelles,
    "../data/processed/Indices finaux/Composites/indice_eq_ref91_norm.tif",
    "idx_eq_ref91"
)

In [ ]:
# Calcul idx_sol mediane
parcelles = add_zonal_mean(
    parcelles,
    "../data/processed/Indices finaux/Sol_norm/Sol_index_normalized.tif",
    "idx_sol"
)

In [ ]:
# Calcul idx_climat_gwl2 mediane
parcelles = add_zonal_mean(
    parcelles,
    "../data/processed/Indices finaux/Climat_norm/I_climat_gwl2_ensemble_clip_norm.tif",
    "idx_climat_gwl2"
)

In [ ]:
# Calcul idx_climat_ref91 mediane
parcelles = add_zonal_mean(
    parcelles,
    "../data/processed/Indices finaux/Climat_norm/I_climat_ref91_ensemble_clip_norm.tif",
    "idx_climat_ref91"
)

In [ ]:
# Calcul idx_topo mediane
parcelles = add_zonal_mean(
    parcelles,
    "../data/processed/Indices finaux/Topo_norm /Topo_index_50m_clip_norm.tif",
    "idx_topo"
)

In [ ]:
# Enregistrement des parcelles avec les nouveaux indices
parcelles.to_file(
    "../outputs/2025_parcelles_indices_med.shp",
    driver="ESRI Shapefile"
)

In [ ]:
parcelles.to_file(
    "../outputs/2025_parcelles_indices_med_gpkg.gpkg",
    layer="parcelles_indices_med",
    driver="GPKG"
)

In [ ]:
# initiation des controles
import geopandas as gpd
parcelles = gpd.read_file("../outputs/2025_parcelles_indices_med_gpkg.gpkg", layer="parcelles_indices_med")

cols = [
    'idx_eq_gwl2', 
    'idx_eq_ref91',
    'idx_sol',
    'idx_climat_gwl2',
    'idx_climat_ref91',
    'idx_topo',
    ]

### 2. Corrélations entre indices

In [ ]:
import matplotlib.pyplot as plt

# 1. Calcul de la corrélation Spearman
corr_matrix = parcelles[cols].corr(method="spearman")
corr_idx_eq = corr_matrix["idx_eq_gwl2"].drop("idx_eq_gwl2")

# 2. Remplacer les noms de colonnes par des labels français
labels_fr = {
    "idx_sol": "Sol",
    "idx_climat_gwl2": "Climat GWL2",
    "idx_climat_ref91": "Climat Ref91",
    "idx_topo": "Topographie",
    "idx_eq_ref91": "Poids éq. Ref91",
}

corr_idx_eq.index = [labels_fr.get(col, col) for col in corr_idx_eq.index]

# 3. Création du barplot avec bleu pastel
plt.figure(figsize=(7,5))
bars = corr_idx_eq.sort_values().plot(kind="bar", color="#A3C1DA")  # bleu pastel

plt.ylabel("Coefficient de corrélation de Spearman")
plt.xlabel("Indice")
plt.title("Corrélation des indices explicatifs avec l'indice composite poids éq. GWL2")

# 4. Annotation des barres avec 2 décimales
for bar in bars.patches:
    height = bar.get_height()
    label = bar.get_x() + bar.get_width() / 2
    name = corr_idx_eq.sort_values().index[bars.patches.index(bar)]
    
    # Si c'est la barre "Poids équivalent REF91", placer la valeur sous le sommet
    if name == "Poids éq. Ref91":
        y_offset = -12  # en points, vers le bas
        va = 'top'
    else:
        y_offset = 3   # habituel, vers le haut
        va = 'bottom'

    bars.annotate(f"{height:.2f}",
                  xy=(bar.get_x() + bar.get_width() / 2, height),
                  xytext=(0, y_offset),
                  textcoords="offset points",
                  ha='center', va=va, fontsize=10)

plt.tight_layout()
plt.show()


### 3. Statistiques régionales

In [ ]:
# Statistiques régionales
import pandas as pd

# Liste des indices
cols_indices = [
    "idx_eq_gwl2",
    "idx_eq_ref91",
    "idx_sol",
    "idx_climat_gwl2",
    "idx_climat_ref91",
    "idx_topo"
]

# Agrégation par région
stats_region = (
    parcelles
    .groupby("Region")[cols_indices]
    .agg(["median", "mean", "std", "count"])
    .reset_index()
)

# Affichage rapide
stats_region.head()


In [ ]:
# CSV des grandeurs statistiques régionales
import pandas as pd

# 1. Agrégation régionale en format long
stats_long = (
    parcelles
    .groupby("Region")[cols_indices]
    .agg(["median", "mean", "std", "count"])
    .stack(level=0)          # empile les indices
    .reset_index()
    .rename(columns={"level_1": "Indice"})
)

# 2. Renommage des indices en français lisible
labels_fr = {
    "idx_eq_gwl2": "Poids équ. GWL2",
    "idx_eq_ref91": "Poids équ. Ref91",
    "idx_sol": "Sol",
    "idx_climat_gwl2": "Climat GWL2",
    "idx_climat_ref91": "Climat Ref91",
    "idx_topo": "Topographie"
}

stats_long["Indice"] = stats_long["Indice"].map(labels_fr)

# 3. Forcer les colonnes statistiques en numérique UNIQUEMENT
cols_stats = ["median", "mean", "std", "count"]

for col in cols_stats:
    stats_long[col] = pd.to_numeric(stats_long[col], errors="coerce")

# 4. Arrondi propre (sauf count si tu veux)
stats_long[["median", "mean", "std"]] = stats_long[["median", "mean", "std"]].round(3)
stats_long["count"] = stats_long["count"].astype("Int64")

# 5. Export CSV propre pour LaTeX
stats_long.to_csv(
    "stats_region_annexe.csv",
    index=False
)

# 6. Vérification rapide
stats_long.head(12)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))
sns.boxplot(
    data=parcelles,
    x="Region",
    y="idx_eq_gwl2",
    palette="pastel"
)
plt.xticks(rotation=45)
plt.ylabel("Indice composite médian (idx_eq_gwl2)")
plt.xlabel("Région agricole")
plt.title("Distribution de l'indice composite par région")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Copier le GeoDataFrame parcelles pour ne pas modifier l'original
parcelles_plot = parcelles.copy()

# 1. Renommer la région
parcelles_plot['Region'] = parcelles_plot['Region'].replace(
    {"Jura - Vallée de Jou": "Jura - Vallée de Joux"}
)

# 2. Calculer la médiane par région pour trier
medians = parcelles_plot.groupby("Region")["idx_eq_gwl2"].median().sort_values()

# 3. Créer un ordre pour seaborn
region_order = medians.index.tolist()

# 4. Boxplot avec regions classées
plt.figure(figsize=(12,6))
sns.boxplot(
    data=parcelles_plot,
    x="Region",
    y="idx_eq_gwl2",
    order=region_order,
    palette="pastel"
)
plt.xticks(rotation=45)
plt.ylabel("Indice composite médian (idx_eq_gwl2)")
plt.xlabel("Région agricole")
plt.title("Distribution de l'indice composite par région (régions triées par médiane)")
plt.tight_layout()
plt.show()


### 4. Statistiques pour exploitations RISC

In [ ]:
# CSV des grandeurs statistiques pour exploitations RISC
import pandas as pd

# ============================================================
# 0. PARAMÈTRES
# ============================================================

cols_indices = [
    "idx_eq_gwl2",
    "idx_eq_ref91",
    "idx_sol",
    "idx_climat_gwl2",
    "idx_climat_ref91",
    "idx_topo"
]

col_expl = "No_RISC"   # identifiant exploitation RISC
col_region = "Region"

# ============================================================
# 1. FILTRAGE DES PARCELLES RISC
# ============================================================

parcelles_risc = parcelles[parcelles[col_expl].notna()].copy()

print("Nombre de parcelles RISC :", parcelles_risc.shape[0])
print("Nombre d'exploitations RISC :", parcelles_risc[col_expl].nunique())

# ============================================================
# 2. DÉTERMINATION DE LA RÉGION DOMINANTE PAR EXPLOITATION
# ============================================================

region_dominante = (
    parcelles_risc
    .groupby([col_expl, col_region])
    .size()
    .reset_index(name="n_parcelles")
    .sort_values([col_expl, "n_parcelles"], ascending=[True, False])
    .drop_duplicates(col_expl)
    .drop(columns="n_parcelles")
)

# ============================================================
# 3. STATISTIQUES PAR EXPLOITATION (TOUTES PARCELLES CONFONDUES)
# ============================================================

stats_risc = (
    parcelles_risc
    .groupby(col_expl)[cols_indices]
    .agg(["median", "mean", "std", "count"])
    .stack(level=0)
    .reset_index()
    .rename(columns={"level_1": "Indice"})
)

# ============================================================
# 4. AJOUT DE LA RÉGION DOMINANTE
# ============================================================

stats_risc = stats_risc.merge(
    region_dominante,
    on=col_expl,
    how="left"
)

# ============================================================
# 5. RENOMMAGE DES INDICES (FRANÇAIS LISIBLE)
# ============================================================

labels_fr = {
    "idx_eq_gwl2": "Poids équ. GWL2",
    "idx_eq_ref91": "Poids équ. Ref91",
    "idx_sol": "Sol",
    "idx_climat_gwl2": "Climat GWL2",
    "idx_climat_ref91": "Climat Ref91",
    "idx_topo": "Topographie"
}

stats_risc["Indice"] = stats_risc["Indice"].map(labels_fr)

# ============================================================
# 6. NETTOYAGE ET FORMATAGE
# ============================================================

cols_stats = ["median", "mean", "std", "count"]

for col in cols_stats:
    stats_risc[col] = pd.to_numeric(stats_risc[col], errors="coerce")

stats_risc[["median", "mean", "std"]] = stats_risc[["median", "mean", "std"]].round(3)
stats_risc["count"] = stats_risc["count"].astype("Int64")

# ============================================================
# 7. TRI POUR LECTURE EN ANNEXE
# ============================================================

ordre_indices = [
    "Poids équ. GWL2",
    "Poids équ. Ref91",
    "Sol",
    "Climat GWL2",
    "Climat Ref91",
    "Topographie"
]

stats_risc["Indice"] = pd.Categorical(
    stats_risc["Indice"],
    categories=ordre_indices,
    ordered=True
)

stats_risc = stats_risc.sort_values(
    by=[col_expl, "Indice"]
)

# ============================================================
# 8. EXPORT CSV FINAL
# ============================================================
# renommage de la colonne d'identifiant pour compatibilité LaTeX
stats_risc_renamed = stats_risc.rename(columns={col_expl: "NoRISC"})
stats_risc_renamed["NoRISC"] = stats_risc_renamed["NoRISC"].astype(int)
stats_risc_renamed.to_csv(
    "stats_exploitations_RISC_annexe.csv",
    index=False,
    encoding="utf-8"
)

# ============================================================
# 9. CONTRÔLE FINAL
# ============================================================

stats_risc_renamed.head(15)


### 5. Statistiques sur les cultures
Une agrégation des nombreux types de cultures en catégories est nécessaire pour rendre l'analyse lisible.
Cela est permis à l'aide d'un "mapping" : un dictionnaire de correspondance vers une catégorie.

In [ ]:
# Statistique sur les cultures
list(parcelles['affectatio'].value_counts().items())

# Mapping des cultures vers des catégories générales
mapping_categories = {

    # Prairies et pâturages
    '601 Prairies temporaires': 'Prairies & pâturages',
    '611 Prairies extensives': 'Prairies & pâturages',
    '612 Prairies peu intensives': 'Prairies & pâturages',
    '613 Prairies perm. (fauche)': 'Prairies & pâturages',
    '616 Pâturages attenants': 'Prairies & pâturages',
    '617 Pâturages extensifs': 'Prairies & pâturages',
    '622 Prairie extens. estivage': 'Prairies & pâturages',
    '623 Prairie peu int. estivage': 'Prairies & pâturages',
    '635 Prairies riveraines (sauf pât)': 'Prairies & pâturages',

    # Céréales
    '502 Orge d\'automne': 'Céréales',
    '501 Orge de printemps': 'Céréales',
    '504 Avoine': 'Céréales',
    '505 Triticale': 'Céréales',
    '513 Blé d\'automne': 'Céréales',
    '512 Blé de printemps': 'Céréales',
    '507 Blé fourrager': 'Céréales',
    '514 Seigle': 'Céréales',
    '516 Epeautre': 'Céréales',
    '510 Blé dur': 'Céréales',
    '515 Méteil de céréales pani.': 'Céréales',
    '506 Méteil de céréales four.': 'Céréales',
    '569 Méteil haricots vesces... min 30% lég': 'Céréales',
    '570 Méteil lentilles av cér ou caméline  min 30 %': 'Céréales',
    '572 Bd semées org utiles TO': 'Céréales',

    # Maïs
    '521 Maïs (ensilage et  vert)': 'Maïs',
    '508 Maïs grain': 'Maïs',
    '519 Semences de maïs': 'Maïs',

    # Légumineuses
    '536 Haricots et vesces en grains (ex fév)': 'Légumineuses',
    '568 Lentilles': 'Légumineuses',
    '540 Pois chiches': 'Légumineuses',
    '537 Pois en grains (ex pois prot)': 'Légumineuses',
    '528 Soja': 'Légumineuses',

    # Oléagineux (hors colza)
    '531 Tournesol huile': 'Oléagineux',
    '534 Lin': 'Oléagineux',
    '544 Cameline': 'Oléagineux',

    # Colza (séparé)
    '527 Colza d\'automne huile': 'Colza',
    '591 Colza d\'automne mpr': 'Colza',
    '526 Colza de printemps huile': 'Colza',

    # Betteraves
    '522 Betteraves sucrières': 'Betteraves',
    '523 Betteraves fourragères': 'Betteraves',

    # Pommes de terre
    '524 Pommes de terre': 'Pommes de terre',
    '525 Plants de pommes de terre': 'Pommes de terre',

    # Vigne
    '701 Vigne': 'Vigne',
    '717 Surf. viti. biodiv.': 'Vigne',

    # Arboriculture
    '702 Cult. fruitière (pommes)': 'Arboriculture',
    '703 Cult. fruitière (poires)': 'Arboriculture',
    '704 C. fruit. (fruits noyaux)': 'Arboriculture',
    '705 Baies pluriannuelles': 'Arboriculture',
    '720 Châtaigneraies': 'Arboriculture',
    '723 Pépinière fruits et baies': 'Arboriculture',
    '718 Truffières': 'Arboriculture',

    # Maraîchage et cultures spéciales
    '545 Cult. mar. plein champ': 'Maraîchage & cultures spéciales',
    '554 Cul.horti. plein champ': 'Maraîchage & cultures spéciales',
    '909 Jardin potager': 'Maraîchage & cultures spéciales',
    '706 Pl. arom. méd. pluriannu.': 'Maraîchage & cultures spéciales',
    '553 Pl. arom. méd.annuelles': 'Maraîchage & cultures spéciales',
    '580 Sorgho grains': 'Maraîchage & cultures spéciales',
    '581 Sorgho pl entière': 'Maraîchage & cultures spéciales',
    '548 Sarrasin': 'Maraîchage & cultures spéciales',
    '574 Quinoa': 'Maraîchage & cultures spéciales',
    '575 Chanvre pour graines': 'Maraîchage & cultures spéciales',
    '577 Autre chanvre': 'Maraîchage & cultures spéciales',
}

parcelles['Categorie'] = parcelles['affectatio'].map(mapping_categories).fillna('Autres non définies')
parcelles.to_file(
    "../outputs/2025_parcelles_indices_med_gpkg.gpkg",
    layer="parcelles_indices_med",
    driver="GPKG"
)

In [ ]:
import pandas as pd

# Transformer le dictionnaire en DataFrame
df_mapping = pd.DataFrame(
    mapping_categories.items(),
    columns=["Culture", "Categorie"]
)



# Calcul des effectifs par culture
effectifs = (
    parcelles["affectatio"]
    .value_counts()
    .rename("Effectif")
    .astype("Int64")
)

# Jointure avec le mapping
df_mapping = df_mapping.merge(
    effectifs,
    left_on="Culture",
    right_index=True,
    how="left"
)
# Corriger le type
df_mapping["Effectif"] = (
    df_mapping["Effectif"]
    .fillna(0)
    .astype("int64")
)

# Trier pour une lecture plus claire (optionnel)
df_mapping = df_mapping.sort_values(
    by=["Categorie", "Culture", "Effectif"]
)

# Export enrichi
df_mapping.to_csv(
    "mapping_cultures_categories_effectifs.csv",
    index=False,
    encoding="utf-8"
)


In [ ]:
import pandas as pd    
csv_table = pd.read_csv("mapping_cultures_categories_effectifs.csv")
print(csv_table.to_latex(index=False))


In [ ]:
# statistiques selon type de culture
import pandas as pd

# ============================================================
# 0. PARAMÈTRES
# ============================================================

col_cat = "Categorie"

cols_indices = [
    "idx_eq_gwl2",
    "idx_eq_ref91",
    "idx_sol",
    "idx_climat_gwl2",
    "idx_climat_ref91",
    "idx_topo"
]

# ============================================================
# 1. FILTRAGE DE SÉCURITÉ
# ============================================================

df = parcelles.copy()

# garder uniquement les parcelles avec une catégorie définie
df = df[df[col_cat].notna()]

# ============================================================
# 2. STATISTIQUES PAR CATÉGORIE (FORMAT LONG)
# ============================================================

stats_cultures = (
    df
    .groupby(col_cat)[cols_indices]
    .agg(["median", "mean", "std", "count"])
    .stack(level=0)              # empile les indices
    .reset_index()
    .rename(columns={"level_1": "Indice"})
)

# ============================================================
# 3. RENOMMAGE LISIBLE DES INDICES
# ============================================================

labels_indices = {
    "idx_eq_gwl2": "Poids équ. GWL2",
    "idx_eq_ref91": "Poids équ. Ref91",
    "idx_sol": "Sol",
    "idx_climat_gwl2": "Climat GWL2",
    "idx_climat_ref91": "Climat Ref91",
    "idx_topo": "Topographie"
}

stats_cultures["Indice"] = stats_cultures["Indice"].map(labels_indices)

# ============================================================
# 4. NETTOYAGE ET FORMATAGE
# ============================================================

cols_stats = ["median", "mean", "std", "count"]

for col in cols_stats:
    stats_cultures[col] = pd.to_numeric(stats_cultures[col], errors="coerce")

stats_cultures[["median", "mean", "std"]] = (
    stats_cultures[["median", "mean", "std"]].round(3)
)

stats_cultures["count"] = stats_cultures["count"].astype("Int64")

# ============================================================
# 5. TRI POUR LECTURE (ANNEXE)
# ============================================================

ordre_indices = [
    "Poids équ. GWL2",
    "Poids équ. Ref91",
    "Sol",
    "Climat GWL2",
    "Climat Ref91",
    "Topographie"
]

stats_cultures["Indice"] = pd.Categorical(
    stats_cultures["Indice"],
    categories=ordre_indices,
    ordered=True
)

stats_cultures = stats_cultures.sort_values(
    by=["count", "Indice"],
    ascending=[False, True]
)

# ============================================================
# 6. EXPORT CSV POUR ANNEXE
# ============================================================

stats_cultures.to_csv(
    "stats_cultures_annexe.csv",
    index=False
)

# ============================================================
# 7. CONTRÔLE RAPIDE
# ============================================================

print("Nombre de catégories :", stats_cultures[col_cat].nunique())
stats_cultures.head(15)


In [ ]:
# Boxplots des indices selon les catégories de culture
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

col_cat = "Categorie"
min_n = 30  # seuil minimum de parcelles par catégorie
cols_indices = [
    "idx_eq_gwl2",
    "idx_eq_ref91",
    "idx_sol",
    "idx_climat_gwl2",
    "idx_climat_ref91",
    "idx_topo"
]
# Copie de travail
df_plot = parcelles.copy()

# Filtrer les catégories trop peu représentées
counts = df_plot[col_cat].value_counts()
categories_valides = counts[counts >= min_n].index
df_plot = df_plot[df_plot[col_cat].isin(categories_valides)]

labels_indices = {
    "idx_eq_gwl2": "Poids équ. GWL2",
    "idx_eq_ref91": "Poids équ. Ref91",
    "idx_sol": "Sol",
    "idx_climat_gwl2": "Climat GWL2",
    "idx_climat_ref91": "Climat Ref91",
    "idx_topo": "Topographie"
}

ordre_reference = (
    df_plot
    .groupby(col_cat)["idx_eq_gwl2"]
    .median()
    .sort_values()
    .index
    .tolist()
)


def boxplot_par_indice(
    df,
    indice,
    ordre_categories,
    col_cat="Categorie",
    labels=None
):
    label_indice = labels.get(indice, indice) if labels else indice

    plt.figure(figsize=(13,6))

    sns.boxplot(
        data=df,
        x=col_cat,
        y=indice,
        order=ordre_categories,
        palette="pastel",
        showfliers=False
    )

    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Catégorie culturale")
    plt.ylabel(label_indice)
    plt.title(
        f"Distribution de l’indice « {label_indice} » selon les catégories culturales"
        #"(ordre fixé selon l’indice composite pondéré GWL2)"
    )

    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
for indice in cols_indices:
    boxplot_par_indice(
        df_plot,
        indice,
        ordre_categories=ordre_reference,
        col_cat=col_cat,
        labels=labels_indices
    )



### 6. Tests statistiques
Permettent de discuter de la significativité des résultats et de l'ampleur des effets observés

#### 6.a Statistiques descritives par groupe

In [ ]:
# Importation des données
import pandas as pd
import scipy.stats as stats
import geopandas as gpd

parcelles = gpd.read_file("../outputs/2025_parcelles_indices_med_gpkg.gpkg", layer="parcelles_indices_med")
df = parcelles.copy()

In [ ]:
# ANNOVA par régions
# Suppression des valeurs manquantes
# Préparation des données pour l'ANOVA
indices = [
    'idx_eq_gwl2',
    'idx_sol',
    'idx_climat_gwl2',
    'idx_topo'
]

facteurs = {
    'Region': 'Region',
    'Culture': 'Categorie'
}


In [ ]:
df.groupby('Region').size().sort_values(ascending=False)
df.groupby('Categorie').size().sort_values(ascending=False)


In [ ]:
def group_count_to_latex(df, group_col, caption, label):
    table = (
        df.groupby(group_col)
          .size()
          .sort_values(ascending=False)
          .reset_index(name='Nombre')
    )

    latex = table.to_latex(
        index=False,
        caption=caption,
        label=label,
        float_format="%.0f"
    )

    print("\n" + "="*80)
    print(latex)
    print("="*80 + "\n")


In [ ]:
group_count_to_latex(
    df,
    group_col="Region",
    caption="Nombre d’observations par région",
    label="tab:region"
)

group_count_to_latex(
    df,
    group_col="Categorie",
    caption="Nombre d’observations par catégorie",
    label="tab:categorie"
)

In [ ]:
def group_std_to_latex(df, group_col, numeric_col='idx_eq_gwl2', caption=None, label=None):
    """
    Calcule l'écart-type d'une colonne numérique par group_col
    et imprime un tableau LaTeX prêt à copier.
    
    Parameters:
    - df : DataFrame
    - group_col : colonne de regroupement (str)
    - numeric_col : colonne numérique par défaut 'idx_eq_gwl2'
    - caption : titre du tableau LaTeX
    - label : label du tableau LaTeX
    """
    # Calcul de l'écart-type
    table = (
        df.groupby(group_col)[numeric_col]
          .std()
          .sort_values(ascending=False)
          .reset_index(name='Écart-type')
    )

    # Caption et label par défaut
    if caption is None:
        caption = f"Écart-type de {numeric_col} par {group_col}"
    if label is None:
        label = f"tab:std_{group_col.lower()}"

    # Conversion en LaTeX
    latex = table.to_latex(
        index=False,
        float_format="%.3f",
        caption=caption,
        label=label
    )

    print("\n" + "="*80)
    print(latex)
    print("="*80 + "\n")

    print("="*80 + "\n")


In [ ]:
# Écart-type par Région
group_std_to_latex(df, group_col="Region")

# Écart-type par Catégorie
group_std_to_latex(df, group_col="Categorie")

In [ ]:
df.groupby('Region')['idx_eq_gwl2'].std()
#df.groupby('Categorie')['idx_eq_gwl2'].std()

#### 6.b ANOVA et Eta squared

In [ ]:
import pandas as pd
import scipy.stats as stats

# --- 1. Fonction ANOVA one-way ---
def anova_oneway(df, valeur, groupe):
    """
    Fait une ANOVA one-way pour une colonne numérique (valeur) 
    en fonction d'un facteur (groupe).
    Retourne F, p, df_between, df_within
    """
    data = df[[valeur, groupe]].dropna()
    groupes = [data[data[groupe] == g][valeur] for g in data[groupe].unique()]

    F, p = stats.f_oneway(*groupes)
    
    # Calcul des degrés de liberté
    df_between = len(groupes) - 1
    df_within = len(data) - len(groupes)

    return F, p, df_between, df_within


# --- 2. Fonction eta squared ---
def eta_squared(F, df_between, df_within):
    return (F * df_between) / (F * df_between + df_within)


# --- 3. Boucle sur indices et facteurs ---
def anova_eta_table(df, indices, facteurs):
    """
    df : DataFrame
    indices : liste de colonnes numériques sur lesquelles faire ANOVA
    facteurs : dict nom_facteur -> colonne
    """
    resultats = []

    for indice in indices:
        for nom_facteur, col_facteur in facteurs.items():
            F, p, df_between, df_within = anova_oneway(df, indice, col_facteur)
            eta2 = eta_squared(F, df_between, df_within)
            
            resultats.append({
                'Indice': indice,
                'Facteur': nom_facteur,
                'F_stat': F,
                'p_value': p,
                'eta_squared': eta2
            })

    return pd.DataFrame(resultats)


# --- 4. Conversion en LaTeX ---
def anova_to_latex(df_anova, caption="Résultats ANOVA", label="tab:anova"):
    """
    Exporte un DataFrame ANOVA en LaTeX :
    - F_stat et eta_squared : exactement 2 chiffres après la virgule
    - p_value : notation scientifique
    """
    df_latex = df_anova.copy()

    # Formater F et eta² à 2 décimales fixes
    df_latex['F_stat'] = df_latex['F_stat'].apply(lambda x: f"{x:.2f}")
    df_latex['eta_squared'] = df_latex['eta_squared'].apply(lambda x: f"{x:.2f}")

    # p-values en notation scientifique
    df_latex['p_value'] = df_latex['p_value'].apply(lambda x: f"{x:.2e}")

    latex = df_latex.to_latex(
        index=False,
        caption=caption,
        label=label
    )

    print("\n" + "="*80)
    print(latex)
    print("="*80 + "\n")


# --- 5. Exemple d'utilisation ---

# indices : liste de colonnes numériques
# facteurs : dict {nom_facteur: colonne_df}
# Ex:
# indices = ['indice1', 'indice2', 'indice3']
# facteurs = {'Region': 'Region', 'Categorie': 'Categorie'}

anova_df = anova_eta_table(df, indices, facteurs)
anova_to_latex(anova_df)


#### 6.c Test de Tukey HSD post-hoc car ANOVA trop significatif pour toutes les variables

In [ ]:
# def group extractor for Tukey
import numpy as np

def extract_groups_for_tukey(
    df,
    value_col,
    group_col,
    groupes=None,
    min_n=2
):
    data = df[[value_col, group_col]].dropna()

    if groupes is not None:
        data = data[data[group_col].isin(groupes)]

    group_names = data[group_col].unique().tolist()

    groups = []
    valid_names = []

    for g in group_names:
        vals = data.loc[data[group_col] == g, value_col].values
        if len(vals) >= min_n:
            groups.append(vals)
            valid_names.append(g)

    if len(groups) < 2:
        raise ValueError(
            f"Tukey HSD nécessite ≥2 groupes valides. "
            f"Groupes trouvés: {valid_names}"
        )

    return groups, valid_names


In [ ]:
groups_region, group_names_region = extract_groups_for_tukey(
    df=parcelles,
    value_col="idx_eq_gwl2",
    group_col="Region"
    
)
print(len(groups_region), group_names_region)

from scipy.stats import tukey_hsd

res = tukey_hsd(*groups_region)
print(res)

In [ ]:
groups_region, group_names_region = extract_groups_for_tukey(
    df=parcelles,
    value_col="idx_eq_gwl2",
    group_col="Region",
    groupes=[   
        "Le Chablais",
        "Préalpes",
        "Plaine de l'Orbe",
        "Jura - Chasseron",
        "Lavaux / Riviera",
        "La Broye",
        "Jura - Vallée de Jou",
        "La Côte",
        "Pied du Jura",
        "Le Jorat",
        "Nord Vaudois",
        "Lausanne et environs",
        "La Venoge",
        "Gros de Vaud"
    
    ]
)
print(len(groups_region), group_names_region)

from scipy.stats import tukey_hsd

res = tukey_hsd(*groups_region)
print(res)

In [ ]:
# Tukey pour cultures

groups_cultures, group_names_cultures = extract_groups_for_tukey(
    df=parcelles,
    value_col="idx_eq_gwl2",
    group_col="Categorie",
    
)
print(len(groups_cultures), group_names_cultures)


from scipy.stats import tukey_hsd

res = tukey_hsd(*groups_cultures)
print(res)

In [ ]:
import pandas as pd

def tukey_to_dataframe(res, group_names):
    stat = res.statistic
    pval = res.pvalue
    ci = res.confidence_interval()

    n = stat.shape[0]  # taille réelle du test

    if len(group_names) != n:
        raise ValueError(
            f"Incohérence : {len(group_names)} noms de groupes, "
            f"mais résultat Tukey de taille {n}"
        )

    rows = []

    for i in range(n):
        for j in range(i + 1, n):
            rows.append({
                "Groupe_1": group_names[i],
                "Groupe_2": group_names[j],
                "Diff_moyenne": stat[i, j],
                "IC_inf": ci.low[i, j],
                "IC_sup": ci.high[i, j],
                "p_value": pval[i, j]
            })

    return pd.DataFrame(rows)


In [ ]:
tukey_regions_df = tukey_to_dataframe(res, group_names_region)
print(tukey_regions_df)


In [ ]:
tukey_cultures_df = tukey_to_dataframe(res, group_names_cultures)
print(tukey_cultures_df)

In [ ]:
def tukey_to_latex(
    df,
    caption,
    label,
    float_fmt="%.3f",
    p_fmt="%.1e"
):
    df_out = df.copy()

    # Formatage propre des colonnes numériques
    df_out["Diff_moyenne"] = df_out["Diff_moyenne"].map(lambda x: float_fmt % x)
    df_out["IC_inf"] = df_out["IC_inf"].map(lambda x: float_fmt % x)
    df_out["IC_sup"] = df_out["IC_sup"].map(lambda x: float_fmt % x)
    df_out["p_value"] = df_out["p_value"].map(lambda x: p_fmt % x)

    # Renommage pour LaTeX
    df_out = df_out.rename(columns={
        "Groupe_1": "Groupe 1",
        "Groupe_2": "Groupe 2",
        "Diff_moyenne": r"$\Delta$ moyenne",
        "IC_inf": "IC$_{inf}$",
        "IC_sup": "IC$_{sup}$",
        "p_value": "$p$-value"
    })

    latex = df_out.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        column_format="llllll"
    )

    print(latex)


In [ ]:
tukey_to_latex(
    tukey_regions_df,
    caption="Résultats du test post-hoc de Tukey (HSD) entre régions agricoles pour l’indice à poids équivalents GWL2",
    label="tab:tukey_regions_idx_eq_gwl2"
)


In [ ]:
tukey_to_latex(
    tukey_cultures_df,
    caption="Résultats du test post-hoc de Tukey (HSD) entre cultures pour l’indice à poids équivalents GWL2",
    label="tab:tukey_cultures_idx_eq_gwl2"
)